# Multi-Goal Financial Asset Recommender System - MVP- **[TEAM MEMBER A] Data Engineering & PyTorch ML**: Web Scraping, Dynamic DataFrame Preprocessing (One-Hot & NaN filling natively to CSV), Masked Weighted PyTorch Autoencoder.- **[TEAM MEMBER B] Scoring & Allocation**: Similarity Score, Dynamic K Filter, Softmax.- **[TEAM MEMBER C] SORR Simulation**: Evaluation loops, Path-Dependent withdrawals, GFR, ETV.- **[UNIFIED DASHBOARD]**: Centralized Config and Hyperparameter Search.### ArchitectureAll function definitions live in worker modules (`_*.py`). This notebook is a **thin orchestration layer** — edit `_constants.py` for config, dive into workers for implementation details.

In [ ]:
%pip install yfinance matplotlib seaborn scipy lxml html5lib requests tqdm nbformat torch
from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
%matplotlib inline

## [TEAM MEMBER A] Layer 1: Data Scraping & PyTorch Embeddings**Step 1:** Configure the pipeline, fetch the asset universe, and build the dataset.- Edit `_constants.py → DEFAULT_PIPELINE_CONFIG` for default settings.- Override specific keys below for this run.- Definitions: `_data_worker.py → fetch_macro_universe(), generate_dataset_member_a()`

In [ ]:
# ── Configure Pipeline ──
from _constants import DEFAULT_PIPELINE_CONFIG, DataSyncMode
config = DEFAULT_PIPELINE_CONFIG.copy()
config["data_source_mode"] = DataSyncMode.OFFLINE_CSV_ONLY  # Override for this run

# ── Fetch Universe & Build Dataset ──
# Definitions in: _data_worker.py
from _data_worker import fetch_macro_universe, generate_dataset_member_a, run_data_diagnostics

tickers = fetch_macro_universe() if config["data_source_mode"] != DataSyncMode.OFFLINE_CSV_ONLY else []
master_df, price_matrix, volume_matrix, daily_returns, drip_returns = generate_dataset_member_a(tickers, config)

# ── Validate Data Quality ──
run_data_diagnostics(master_df, config)

**Step 2:** Train the PyTorch embedding model (autoencoder with multi-horizon targets).- Definitions: `_ml_worker.py → AssetEmbeddingNet, train_pytorch_embedding_model()`

In [ ]:
# ── Train Embedding Model ──
# Definitions in: _ml_worker.py
from _ml_worker import train_pytorch_embedding_model

DATA_CACHE = train_pytorch_embedding_model(
    master_df, price_matrix, volume_matrix, daily_returns,
    config,
    drip_daily_returns=drip_returns
)

## [TEAM MEMBER B] Layer 2: Scoring, Dynamic K Selection & AllocationDefinitions: `_scoring_worker.py → build_user_preference_vector(), recommend_and_allocate_member_b()`

## [TEAM MEMBER C] Evaluation Framework & SimulationDefinitions: `_sim_worker.py → evaluate_portfolio_member_c()`

## [PART B] Dynamic Top-K Recommendation — Multi-Profile DemonstrationThis section tests the full pipeline by running **three distinct user profiles** through the system:1. **Conservative Retiree** — Low risk, high near-term withdrawal needs2. **Balanced Growth** — Moderate risk, medium-term goals3. **Aggressive Young Investor** — High risk, long-term growth focusFor each profile, the system:- Constructs a **user preference vector** from the trained embedding space- Scores the entire asset universe via **cosine similarity + volatility penalty**- Applies **Dynamic K thresholding** (θ) to select the optimal asset set- Computes **temperature-scaled softmax weights** for allocation- Runs **historical rolling-window backtesting** (Member C evaluation)- Visualizes the results with score distributions, sector breakdowns, and weight charts

In [ ]:
# ── Multi-Profile Dynamic Top-K Test ──
from _constants import TEST_PROFILES
from _scoring_worker import build_user_preference_vector, recommend_and_allocate_member_b
from _display_worker import display_dynamic_top_k
from _sim_worker import evaluate_portfolio_member_c

all_profile_results = []

for profile in TEST_PROFILES:
    print("\n" + "\u2550" * 80)
    print(f"  TESTING PROFILE: {profile['profile_name']}")
    print("\u2550" * 80)

    # Step 1: Build user preference vector from profile inputs
    user_vec = build_user_preference_vector(DATA_CACHE, profile, config)

    # Step 2: Score, filter, and allocate (reads theta/weights from config)
    recs = recommend_and_allocate_member_b(
        dataset=DATA_CACHE,
        user_profile=profile,
        user_vector=user_vec,
        config=config
    )

    # Step 3: Display comprehensive results
    stats = display_dynamic_top_k(DATA_CACHE, recs, profile, user_vec)

    # Step 4: Run simulation evaluation (Member C)
    sim_metrics = evaluate_portfolio_member_c(DATA_CACHE, recs, profile, config=config)

    print(f"  \u2500\u2500\u2500 SIMULATION RESULTS \u2500\u2500\u2500")
    print(f"  Goal Fulfillment Rate (GFR): {sim_metrics['GFR']:.2%}")
    print(f"  Expected Terminal Value:     ${sim_metrics['ETV']:,.0f}")
    print(f"  Objective Score:             {sim_metrics['Objective_Function_Score']:,.0f}")
    print(f"  Total Simulations Run:       {sim_metrics['Total_Simulations']}")

    all_profile_results.append({
        "Profile": profile["profile_name"],
        "Risk": profile["risk_tolerance"],
        "Start Cap": f"${profile['start_cap']:,.0f}",
        "K (Assets)": stats["k"],
        "HHI": f"{stats['hhi']:.0f}",
        "Wtd Vol": f"{stats['weighted_volatility']:.2%}",
        "GFR": f"{sim_metrics['GFR']:.2%}",
        "ETV": f"${sim_metrics['ETV']:,.0f}",
        "Obj Score": f"{sim_metrics['Objective_Function_Score']:,.0f}"
    })

print("\n\n" + "\u2588" * 80)
print("\u2588  CROSS-PROFILE COMPARISON SUMMARY")
print("\u2588" * 80)
display(pd.DataFrame(all_profile_results))

### Threshold (θ) Sensitivity AnalysisHow does the Dynamic K threshold affect each user profile's portfolio?

In [ ]:
# ── Theta Sensitivity Analysis ──
# Definitions in: _display_worker.py → plot_theta_sensitivity()
from _display_worker import plot_theta_sensitivity

plot_theta_sensitivity(TEST_PROFILES, DATA_CACHE, config, evaluate_fn=evaluate_portfolio_member_c)